*   A multilayer perceptron, or MLP, is a neural network that stacks linear layers with nonlinear activation functions.
*   It is the first model here that can represent nonlinear patterns.

In [2]:
import math
import random
import numpy as np
import torch

# 5.1.1 Hidden Layers

## 1. Intuition



*   A hidden layer is an intermediate layer between the input and output.
> * It is called hidden because its values are not labels and are not directly observed in the dataset.

*   A linear layer alone can only build linear decision boundaries.

*   A hidden layer followed by a nonlinear activation lets the model build more flexible functions.


## 2. Why this exists


*   Many real patterns are not linear.
*   Hidden layers let a model learn intermediate features before making a final prediction.

## 3. Examples

*   Manual Python: 1 hidden unit from 2 inputs.

In [3]:
x = [1.0, 2.0]
w = [0.5, -1.0]
b = 0.1

hidden_raw = x[0] * w[0] + x[1] * w[1] + b # 1*0.5 + 2*-1 + 0.1 = 0.5-2+0.1 = -1.4
hidden = max(0.0, hidden_raw)
hidden # 0.0 > -1.4 so 0.0

0.0

*   PyTorch: a tiny hidden layer maps 2 features to 3 hidden values.
* Notice how `raw_hidden = X @ W.T + b`, not the standard `y = X @ W + b`?
> * Because PyTorch stored `W` directly as shape of `(3, 2)` even though its true shape should be `(2, 3)`.
> * PyTorch thinks: "I have 3 neurons, and each neuron needs 2 weights." Therefore, it stores 1 row per neuron:
    ```
    W.shape = (out_features, in_features)
          = (3, 2)
    ```
> * During the forward pass, PyTorch transposes it:

    ```
    X.shape   = (1, 2)
    W.T.shape = (2, 3)

    X @ W.T   = (1, 3)
    ```




In [4]:
X = torch.tensor([[1.0, 2.0]]) # Shape of (1, 2)
hidden_layer = torch.nn.Linear(2, 3) # Creates a weight tensor of (3, 2) and bias tensor of (3, )
raw_hidden = hidden_layer(X) # Shape of (1, 2) @ (3, 2)T (or 2, 3) + (3,) = shape of (1, 3)

hidden = torch.relu(raw_hidden) # ReLU(x) = max(0, x); if the value is positive → keep it, if the value is negative → replace it with 0

hidden.shape # Always (1, 3) because ReLU changes values, not dimensions

torch.Size([1, 3])

## 4. Step-by-step breakdown

* The manual code first computes a weighted sum.

* `max(0.0, hidden_raw)` is a ReLU activation. It keeps positive values and replaces negative values with zero.

* The PyTorch layer computes 3 hidden weighted sums at once (because `Linear(2, 3)` creates 3 hidden neurons), producing `raw_hidden` with shape `(1, 3)`.

* `torch.relu` applies the same nonlinear rule elementwise.

## 5. Connection to ML systems

*   MLPs stack hidden representations.
*   Later layers use these hidden values instead of the original raw inputs.

### ReLU creates sparsity

This is a big reason ReLU works well.

Imagine a hidden layer with 1,000 neurons.

Before ReLU:

```text
[-2.1, 0.5, -0.7, 3.2, -1.8, ...]
```

After ReLU:

```text
[0, 0.5, 0, 3.2, 0, ...]
```

Many neurons become inactive (zero). This creates a **sparse representation**:

```text
active features:
- edge detected
- wheel-like shape detected
- texture detected

inactive:
- irrelevant patterns
```

The next layer can focus on the features that are currently useful instead of processing every possible signal equally.

---

## Why not keep the negatives?

You might ask:

> Why not pass negative values through? Why throw away information?

The reason is that, for many features, the important question is whether the feature is **present**.

Imagine a neuron represents:

```text
"How much does this input look like a wheel?"
```

A value of:

```text
+5
```

means:

```text
Strong wheel evidence
```

A value of:

```text
-5
```

means:

```text
Not wheel evidence
```

After ReLU:

```text
+5 → 5
-5 → 0
```

The next layer often only needs:

```text
Is the wheel feature active?
```

rather than:

```text
How strongly does this example disagree with being a wheel?
```

If the network needs the opposite signal, other neurons can learn that pattern.

## 6. Common confusion points

- Hidden does not mean mysterious; it means intermediate.
- A hidden layer without a nonlinear activation (e.g. ReLU) still collapses into 1 linear transformation. 2 linear layers are effectively just 1 bigger linear layer.
- Hidden width is the number of hidden units (or # of `out_features`).
- More hidden units increase the model's capacity (flexibility) to learn complex, non-linear relationships.
> * Hidden layers provide capacity; ReLU provides expressiveness. Together they make deep learning possible.
> * However, this also increases the number of weights, memory usage, computation, and the risk of overfitting (learning noise in the training data).

# 5.1.2 Activation Functions

## 1. Intuition

* An activation function is a nonlinear function applied to layer outputs.

* Common activation functions include `ReLU`, `sigmoid`, and `tanh`.

* Nonlinear means the function is not just a weighted sum or constant scaling.

## Common activation functions

Activation functions all have the same job:

> Take a neuron's weighted sum (`X @ W + b`) and transform it before passing it to the next layer.

They differ in **how** they transform the value and where they're most useful.

| Activation | Formula | Output Range | Intuition | Common Use |
|------------|---------|--------------|-----------|------------|
| **ReLU** | $ReLU(x)=\max(0,x)$ | $[0,\infty)$ | Keep positive evidence, discard negative evidence | Hidden layers in most modern neural networks |
| **Sigmoid** | $\sigma(x)=\frac{1}{1+e^{-x}}$ | $(0,1)$ | Squash values into a probability | Binary classification output |
| **tanh** | $\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $(-1,1)$ | Squash values while preserving the sign | Older hidden layers, recurrent neural networks (RNNs) |

---

### 1. ReLU (Rectified Linear Unit)

$$
ReLU(x)=\max(0,x)
$$

Examples:

```text
-3 → 0
-1 → 0
 0 → 0
 2 → 2
 5 → 5
```

**Pros**

- Simple and computationally efficient.
- Creates sparse activations (many neurons become inactive).
- Avoids many optimization problems that affect sigmoid and tanh.

**Common uses**

- Hidden layers in MLPs.
- Convolutional Neural Networks (CNNs) for computer vision.
- Many transformer feed-forward networks (modern LLMs often use variants like **GELU** or **SiLU**, which are smoother than ReLU).

---

### 2. Sigmoid

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

Examples:

```text
-10 → 0.000
 -2 → 0.119
  0 → 0.500
  2 → 0.881
 10 → 1.000
```

Sigmoid converts any value into the range **0–1**, making it ideal for probabilities.

**Pros**

- Naturally interpretable as a probability.

**Cons**

- Large positive/negative inputs saturate near 0 or 1, causing very small gradients (**vanishing gradients**).

**Common uses**

- Final output layer for **binary classification**.
- Logistic regression.
- Predicting probabilities, such as:
  - Spam vs. not spam.
  - Fraud vs. legitimate transaction.
  - Disease present vs. absent.
  - User clicks vs. doesn't click.

---

### 3. tanh (Hyperbolic Tangent)

$$
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

Examples:

```text
-10 → -1.000
 -2 → -0.964
  0 →  0.000
  2 →  0.964
 10 →  1.000
```

Unlike sigmoid, tanh is centered around zero.

```text
Sigmoid:
0 → 0.5

tanh:
0 → 0
```

This zero-centered output often made optimization easier than sigmoid.

**Pros**

- Preserves whether a signal is positive or negative.
- Zero-centered outputs can make optimization more stable than sigmoid.

**Cons**

- Still suffers from vanishing gradients for very large positive/negative values.

**Common uses**

- Hidden layers in older neural networks.
- Recurrent Neural Networks (RNNs) and LSTMs, where hidden states often need to represent a **continuous internal state** containing both positive and negative information.

For example, suppose a hidden state represents **sentiment** while reading a sentence:

```text
+1  → strongly positive
 0  → neutral
-1  → strongly negative
```

A zero-centered activation like **tanh** naturally represents all three cases. By contrast, a sigmoid output is always between **0 and 1**, making it better suited for probabilities than representing opposing states.

More generally, tanh is useful whenever a hidden state benefits from encoding both the **direction (sign)** and **strength (magnitude)** of a signal. For example:

```text
Sentiment:
-1 ←──────── 0 ────────→ +1

negative       neutral      positive
```

```text
Emotion:
-1 ←──────── 0 ────────→ +1

sad            neutral      happy
```

```text
Agreement:
-1 ←──────── 0 ────────→ +1

disagree       unsure       agree
```

In LSTMs specifically, **tanh does not decide what to remember or forget**. The **sigmoid gates** make those decisions:

- 0 → forget this information.
- 1 → keep this information.

The **tanh** activation instead represents the **content** being stored, allowing the memory to contain both positive and negative values.

---

## Rule of thumb

- **ReLU**: Default choice for hidden layers in modern deep learning.
- **Sigmoid**: Use when the output should represent a probability between **0 and 1** (binary classification).
- **tanh**: Use when hidden states benefit from being centered around zero, especially in recurrent networks.

> **Modern note:** LLMs typically don't use plain ReLU anymore. Most transformer models use smoother activation functions such as **GELU** (e.g., BERT, GPT-2) or **SiLU/Swish** (e.g., Llama), which tend to train more effectively while serving the same general purpose of introducing non-linearity.

---

## A good mental model:

| Activation | Mental model |
|---|---|
| ReLU | Hard switch: "on/off" |
| Leaky ReLU | ReLU but let a tiny amount leak through |
| GELU | Smooth probabilistic gate |
| SiLU | Smooth sigmoid gate; a combination of leaky + GELU|

## 2. Why this exists

*   Without activation functions, stacking linear layers is still equivalent to 1 linear layer.
*   Activations are what let deep networks represent nonlinear patterns.

## 3. Examples

*   Compare 3 activation functions on tiny inputs.

In [5]:
x = torch.tensor([-2.0, 0.0, 2.0])

relu = torch.relu(x)
sigmoid = torch.sigmoid(x) # Distributed from [0, 1]
tanh = torch.tanh(x) # Distributed from [-1, 1]
relu, sigmoid, tanh

(tensor([0., 0., 2.]),
 tensor([0.1192, 0.5000, 0.8808]),
 tensor([-0.9640,  0.0000,  0.9640]))

*   A small MLP forward pass.

In [6]:
X = torch.tensor([[1.0, -1.0]]) # shape of (1, 2)
W1 = torch.randn(2, 3)
b1 = torch.zeros(3)
H = torch.relu(X @ W1 + b1) # (1, 2) @ (2, 3) + (3,) = shape of (1, 3)
H

tensor([[0.0000, 0.0000, 1.9504]])

## 4. Step-by-step breakdown

* ReLU outputs zero for negative inputs and keeps positive inputs.

* Sigmoid maps inputs into the range from 0 to 1.

* Tanh maps inputs into the range from -1 to 1.

* The MLP example computes hidden pre-activations with `X @ W1 + b1`, then applies ReLU.

## 5. Connection to ML systems

*   Modern MLPs commonly use ReLU-like activations because they are simple and train well in many settings.

## 6. Common confusion points

- Activation functions are applied elementwise.
- Sigmoid and tanh can saturate when inputs are very large or very small, causing gradients to become extremely small (vanishing gradients).
- ReLU can output exact zeros.
- Nonlinearity is essential for stacked layers to add expressive power to learn more relationships.

# 5.1.3 Summary and Discussion

## 1. Intuition

*   An MLP combines linear layers and activation functions.

*   The simplest MLP has an input layer, one hidden layer, an activation, and an output layer.

## 2. Why this exists

*   MLPs are a general-purpose starting point for learning nonlinear relationships in vector data.

## 3. Examples

*   A compact 2-layer MLP.

In [8]:
mlp = torch.nn.Sequential(
    torch.nn.Linear(2, 3), # weight shape: (3, 2), so weight.T (2, 3)
    torch.nn.ReLU(),
    torch.nn.Linear(3, 1), # weight shape: (1, 3), so weight.T (3, 1)
)
mlp(torch.tensor([[1.0, 2.0]])).shape # (1, 2) @ (2, 3) = (1, 3) -> (1, 3) @ (3, 1) = (1, 1)

torch.Size([1, 1])

## 4. Step-by-step breakdown

*   `Sequential` runs modules in order.

*   The first linear layer maps 2 inputs to 3 hidden values.

*   `ReLU` applies nonlinearity.

*   The final linear layer maps 3 hidden values to 1 output.

## 5. Connection to ML systems

*   The MLP pattern appears throughout deep learning.
*   Later architectures change the layer type, but the idea of learned representations remains.

## 6. Common confusion points

- Layer order matters.
- Hidden size controls representation width.
- Output size should match the task.
- The model is untrained until parameters are fit with data.

# 5.1.4 Exercises

## 1. Intuition

*   These exercises check whether you can reason about MLP shapes and activations.

## 2. Why this exists

*   Shape tracking is the first defense against confusing MLP code.

## 3. Examples

*   Exercise 1: create an MLP that maps 4 inputs to 2 outputs.
> * Input features are determined by the first `Linear` layer.
> * Output features are determined by the last `Linear` layer.
> * Each layer transforms the features it receives, passing its output to the next layer until the network produces its final output.
> * In an MLP, `Linear` layers change the feature dimension, while activation functions such as `ReLU` preserve it.
> * An MLP essentially recycles through `Linear` and `ReLU` across multiple layers.

In [9]:
net = torch.nn.Sequential(
    torch.nn.Linear(4, 5), # This MLP expects 4 inputs features
    torch.nn.ReLU(),
    torch.nn.Linear(5, 2), # This MLP produces 2 inputs features
)
net(torch.randn(3, 4)).shape # (3, 4) @ (4, 5) = (3, 5) -> (3, 5) @ (5, 2) -> (3, 2)

torch.Size([3, 2])

*   Exercise 2: inspect ReLU on negative and positive values.


In [10]:
x = torch.tensor([-1.0, 0.0, 1.0])
torch.relu(x) # [0, 0, 1] as negatives are converted to 0

tensor([0., 0., 1.])

## 4. Step-by-step breakdown

* Exercise 1 checks batch size, input size, hidden size, and output size.
> * The batch has 3 examples, so the output has 3 rows.

* Exercise 2 checks the ReLU rule directly.

## 5. Connection to ML systems

*   These checks prepare for implementing MLP training loops.

## 6. Common confusion points

- The final layer output count is task-dependent.
- ReLU does not change tensor shape.
- Batch size (e.g. inference tensor's 1st dimension) is independent of layer width, or feature dimensions.
- Hidden layers store learned intermediate representations.